In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


FILE_PATH = "../data/가상Tech_업무가이드.pdf"
loader = PyPDFLoader(FILE_PATH) #로더 설정

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap = 50)

documents = loader.load_and_split(text_splitter=text_splitter)
embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(documents, embeddings)
retriever = db.as_retriever(search_type="mmr", search_kwargs = {"k":3, "fetch_k":10})

questions=[
"연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
"서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
"출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
"회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
"재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
"직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
"출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
"개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
"업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
"제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.",
]


In [6]:
for question in questions:
    docs = retriever.invoke(question)
    print(f"{question}  \n\n")
    for doc in docs:
        print(f"{doc.page_content}\n")
        print("--"*30)
    print("=="*30)


연차를 사용하려면 최소 며칠 전에 신청해야 하나요?  


2년
소유자 확인 후 휴지통 이동
계약·세금 증빙
10년
법무/재무 승인 후 파기
지원 티켓
3년
개인정보 마스킹 후 통계만 보존
장애 회고
영구
서비스 종료 시 아카이브
검색 품질 팁
문서 제목과 본문에 실제 사용자가 질문할 표현을 포함합니다. ‘MFA’를 설명하는
문서라면 ‘다중 인증’, ‘2단계 인증’, ‘인증 앱’도 함께 적습니다.

------------------------------------------------------------
• 프로젝트 채널 이름은 #prj-프로젝트명, 운영 채널은 #ops-서비스명 형식을 사용합니다.
 
3.2 메시지와 회의 예절
 • 메시지는 배경-요청-기한 순서로 씁니다. 단순 확인에는 이모지 반응을 활용합니다.
 • 긴급이 아닌 업무는 19:00 이후 즉시 응답을 기대하지 않습니다. 예약 전송을 권장합니다.
 • 회의 초대에는 목적, 사전 자료, 의사결정 항목을 포함합니다. 참석자는 최소화합니다.
 • 외부 회의 녹화는 참석자 전원의 동의를 받은 뒤 시작하며, 녹화본은 30일 후 삭제합니다.
 
공식 결정
Slack 대화만으로 정책·일정·범위를 확정하지 않습니다. 결정자는 Notion 의사결정
로그 또는 Jira 이슈에 결론과 근거를 기록합니다.

------------------------------------------------------------
가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
08
CHAPTER 07
휴가·비용·복지
예측 가능한 신청 절차로 개인의 휴식과 팀 운영을 함께 지킵니다.
 
7.1 휴가
 구분
 신청 시점
 승인·증빙
연차
1일 전
팀장 승인, 3일 이상은 2주 전 협의
반차
당일 10:00 전
팀장 승인
병가
가능한 즉시
3일 초과 시 진료 확인서 제출
경조휴가
사유 확인 즉시
People Ops 확인, 사내 규정 일수 적용
7.2 비용 처리
 • 법인카드 사용 내역은 영수증

In [7]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = PromptTemplate.from_template(
    "{docs}에 {question}에 관한 내용이 없으면 '관련 내용이 없습니다.'고 말해주세요."
    "{docs}에서 {question}에 관해 찾아서 설명해주세요."
)

output_parser = StrOutputParser()
chain = prompt | model | output_parser
answers = chain.batch([{"docs":retriever.invoke(question),"question":question} for question in questions],config = {"max_concurrency" : 3},)

In [8]:
for i,answer in enumerate(answers):
    print(f"[{i+1}]. {questions[i]} ?")
    print(">>"*50)
    print(answer)
    print("=="*50,end="\n\n")

[1]. 연차를 사용하려면 최소 며칠 전에 신청해야 하나요? ?
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
연차를 사용하려면 최소 1일 전에 신청해야 하며, 3일 이상의 연차를 사용할 경우에는 2주 전 협의가 필요합니다.

[2]. 서울 출장 시 숙박비는 최대 얼마까지 지원되나요? ?
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
서울 출장 시 숙박비는 1박당 최대 15만원(세금 포함)까지 지원됩니다.

[3]. 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요? ?
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
출장을 다녀온 후 비용 정산은 지출일로부터 5영업일 안에 영수증과 목적을 ExpenseHub에 등록해야 합니다.

[4]. 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요? ?
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
관련 내용이 없습니다.

[5]. 재택근무를 신청하려면 어떤 조건을 충족해야 하나요? ?
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
재택근무를 신청하려면 다음 조건을 충족해야 합니다:

1. **주 2회 기준**: 재택근무는 주 2회로 제한되어 있으며, 매주 월요일까지 캘린더에 'WFH' (Work From Home) 표시를 해야 합니다.

이 외에 추